# Medical Insurance Cost Prediction

## Step 1: Problem Definition

### Business Objective

The objective of this project is to build a Machine Learning model that can estimate medical insurance charges based on customer-related features such as age, BMI, number of children, smoking status, sex, and region. The predicted charges can provide an estimated cost before the final insurance charge is determined. This can help make the cost estimation process faster, more consistent, and data-driven.

## Step 2: Libraries Import

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## Step 3: Data Loading

In [7]:
Data_Path = Path("../data/insurance.csv")

if not Data_Path.exists():
    raise FileNotFoundError(f"File Not Found: {Data_Path}")

df = pd.read_csv(Data_Path)
print(f"Shape  : {df.shape}")
print(f"Dataset Loaded Successfully!")

Shape  : (1338, 7)
Dataset Loaded Successfully!


## Step 4: Basic Inspection

In [8]:
summary = pd.DataFrame({
    'dtype'      : df.dtypes.astype(str),
    'non_null'   : df.count(),
    'null_count' : df.isnull().sum(),
    'null_pct'   : (df.isnull().sum() / len(df) * 100).round(2),
    'unique'     : df.nunique(),
    'unique_pct' : (df.nunique() / len(df) * 100).round(2),
})
print('\n=== COLUMN SUMMARY ===')
print(summary)


num = df.select_dtypes(include='number') 
num_summary = num.describe().T
num_summary['skew']     = num.skew()
num_summary['kurtosis'] = num.kurtosis()
num_summary['range']    = num.max() - num.min()
num_summary['cv']       = (num.std() / num.mean()).round(3)  # coefficient of variation
print('\n=== NUMERICAL — EXTENDED SUMMARY ===')
print(num_summary.round(3))

print('\n=== CATEGORICAL — VALUE COUNTS ===') 
cat_cols = df.select_dtypes(include=["object", "str", "category"]).columns

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())
    print(df[col].value_counts(normalize=True).round(4) * 100)

# Constant columns (zero variance)
constant_cols = df.columns[df.nunique(dropna=False) == 1].tolist()
print(f'Constant columns      : {constant_cols if constant_cols else "none ✅"}')

# ID-like columns (unique = row count)
id_like = df.columns[df.nunique() == len(df)].tolist()
print(f'ID-like columns       : {id_like if id_like else "none ✅"}')

# High null columns (> 30%)
high_null = df.columns[df.isnull().mean() > 0.3].tolist()
print(f'High null columns     : {high_null if high_null else "none ✅"}')

print('\n=== DOMAIN SANITY ===')
sanity_checks = {
    "Age in [0,120]": df["age"].between(0, 120).all(),
    "BMI in [10,70]": df["bmi"].between(10, 70).all(),
    "Charges >= 0": (df["charges"] >= 0).all(),
    "Children >= 0": (df["children"] >= 0).all(),
    "Sex values valid": set(df["sex"].unique()) <= {"male", "female"},
    "Smoker values valid": set(df["smoker"].unique()) <= {"yes", "no"}
}

for check, result in sanity_checks.items():
    print(f"{check:<22}: {result}")



=== COLUMN SUMMARY ===
            dtype  non_null  null_count  null_pct  unique  unique_pct
age         int64      1338           0       0.0      47        3.51
sex           str      1338           0       0.0       2        0.15
bmi       float64      1338           0       0.0     548       40.96
children    int64      1338           0       0.0       6        0.45
smoker        str      1338           0       0.0       2        0.15
region        str      1338           0       0.0       4        0.30
charges   float64      1338           0       0.0    1337       99.93

=== NUMERICAL — EXTENDED SUMMARY ===
           count       mean        std       min       25%       50%  \
age       1338.0     39.207     14.050    18.000    27.000    39.000   
bmi       1338.0     30.663      6.098    15.960    26.296    30.400   
children  1338.0      1.095      1.205     0.000     0.000     1.000   
charges   1338.0  13270.422  12110.011  1121.874  4740.287  9382.033   

                7

## Data Type Verification

In [12]:
print('\n=== HIDDEN STRING / NUMERIC PATTERN DETECTION ===')

for col in df.select_dtypes(include=['object', 'string', 'category']).columns:

    values = df[col].astype('string')

    has_digits = values.str.contains(r'\d', regex=True, na=False).any()

    has_special_numeric_patterns = values.str.contains(
        r'[$€£%]|,\d',
        regex=True,
        na=False
    ).any()

    if has_digits or has_special_numeric_patterns:
        print(
            f'⚠️ {col:<20}: '
            f'Suspicious numeric pattern found — investigate'
        )
    else:
        print(
            f'✅ {col:<20}: '
            f'No suspicious numeric pattern detected'
        )

print('\n=== ORDINAL vs NOMINAL CLASSIFICATION ===')

nominal_columns = ['sex', 'smoker', 'region']

for col in nominal_columns:
    df[col] = df[col].astype('category')

    print(
        f'{col:<10}: nominal categorical | '
        f'categories = {list(df[col].cat.categories)}'
    )

print('\n=== SEMANTIC + EXPECTED vs ACTUAL DTYPE ===')

type_map = {
    'age':        ('discrete numerical', 'numerical'),
    'sex':        ('nominal categorical', 'categorical'),
    'bmi':        ('continuous numerical', 'numerical'),
    'children':   ('discrete numerical', 'numerical'),
    'smoker':     ('nominal categorical', 'categorical'),
    'region':     ('nominal categorical', 'categorical'),
    'charges':    ('continuous numerical', 'numerical')
}

def dtype_matches(series, expected_type):

    if expected_type == 'numerical':
        return pd.api.types.is_numeric_dtype(series)

    elif expected_type == 'categorical':
        return (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
            or isinstance(series.dtype, pd.CategoricalDtype)
        )

    elif expected_type == 'datetime':
        return pd.api.types.is_datetime64_any_dtype(series)

    elif expected_type == 'boolean':
        return pd.api.types.is_bool_dtype(series)

    return False


for col, (semantic_type, expected_type) in type_map.items():

    actual_dtype = str(df[col].dtype)

    result = dtype_matches(df[col], expected_type)

    status = '✅' if result else '❌'

    print(
        f'{status} {col:<20} | '
        f'Semantic: {semantic_type:<25} | '
        f'Expected: {expected_type:<12} | '
        f'Actual: {actual_dtype}'
    )

print('\n=== HIDDEN NON-NUMERIC VALUES CHECK ===')

numeric_cols = ['age', 'bmi', 'children', 'charges']

for col in numeric_cols:
    converted = pd.to_numeric(df[col], errors='coerce')
    hidden_non_numeric = (converted.isna() & df[col].notna())
    count = hidden_non_numeric.sum()

    if count > 0:
        print(f'⚠️  {col:<20}: {count} non-numeric value(s) found')
    else:
        print(f'✅ {col:<20}: Clean numeric values — OK')




=== HIDDEN STRING / NUMERIC PATTERN DETECTION ===
✅ sex                 : No suspicious numeric pattern detected
✅ smoker              : No suspicious numeric pattern detected
✅ region              : No suspicious numeric pattern detected

=== ORDINAL vs NOMINAL CLASSIFICATION ===
sex       : nominal categorical | categories = ['female', 'male']
smoker    : nominal categorical | categories = ['no', 'yes']
region    : nominal categorical | categories = ['northeast', 'northwest', 'southeast', 'southwest']

=== SEMANTIC + EXPECTED vs ACTUAL DTYPE ===
✅ age                  | Semantic: discrete numerical        | Expected: numerical    | Actual: int64
✅ sex                  | Semantic: nominal categorical       | Expected: categorical  | Actual: category
✅ bmi                  | Semantic: continuous numerical      | Expected: numerical    | Actual: float64
✅ children             | Semantic: discrete numerical        | Expected: numerical    | Actual: int64
✅ smoker               | Semanti